In [1]:
folder_path='G:\My Drive\CS671-DL\AudioLDM\CLAPAutoencoder\Excelfilebaazi-20240426T185415Z-001\Excelfilebaazi'

In [ ]:
# pip ionstall cudf

In [2]:
import os
import pandas as pd
import numpy as np

# List to store vectors from all CSVs
all_vectors = []

# Limit the number of files processed for testing
max_files = 495

# Loop through files in the folder
for Num, file in enumerate(os.listdir(folder_path)):
    vector_concat = []
    if file.endswith('.csv') and Num < max_files:
        file_path = os.path.join(folder_path, file)

        # Read CSV file into a Pandas DataFrame
        df = pd.read_csv(file_path)
        print(Num)
        # Iterate through rows and concatenate values from all columns to form vectors
        for _, row in df.iterrows():
            # Concatenate values from all columns to form a vector
            vector_values = row.values.astype(np.float16)[:1 * 1024]  # Convert to float32
            vector = vector_values.reshape((1, 1024))  # Reshape to the desired shape (1024, 4)
            # Append the vector to the list
            vector_concat.append(vector)

        all_vectors += vector_concat

# Print the first vector from the list for verification
print(all_vectors[0])


0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
240
241
242
243
244
245
246
247
248
249
250
251
252
253
254
255
256
257
258
259
260
261
262
263
264
265
266
267
268
269
270
271
272
273
274
275
276
27

In [3]:
print(all_vectors[0])

[[ 1.598     1.202     0.9385   ... -0.8706    0.006767  1.147   ]]


In [ ]:
# import numpy as np

# # Generate random data points
# num_data_points = 32
# data_size = (4, 1024)
# all_vectors = []

# for _ in range(num_data_points):
#     data_point = np.random.randn(*data_size)  # Generate random data of size (4, 1024)
#     all_vectors.append(data_point)

# # Convert the list to a numpy array if needed
# all_vectors = np.array(all_vectors)

# # Check the shape of the generated data
# print(all_vectors.shape)

In [6]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from torch.utils.data import TensorDataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

# Check if GPU is available and set device accordingly
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Assuming all_vectors is a list of data points where each data point is a numpy array of shape (4, 1024)
# Also assuming folder_path is the specified path for saving the model

# Convert the list of data points to a numpy array and flatten it
data_points = np.array(all_vectors)
# Assuming the input data shape is (num_samples, 4, 1024)
tensor_data = torch.tensor(data_points, dtype=torch.float32).to(device)

# Define a Convolutional Autoencoder model with LayerNorm
class ConvAutoencoder(nn.Module):
    def __init__(self):
        super(ConvAutoencoder, self).__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, stride=1, padding=1),  # Input shape: (1, 4, 1024), Output shape: (16, 4, 1024)
            nn.ReLU(),
            nn.LayerNorm([16, 1, 1024]),  # LayerNorm after first Conv2d layer
            nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1),  # Input shape: (16, 4, 1024), Output shape: (32, 4, 1024)
            nn.ReLU(),
            nn.LayerNorm([32, 1, 1024]),  # LayerNorm after second Conv2d layer
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),  # Input shape: (32, 4, 1024), Output shape: (64, 4, 1024)
            nn.ReLU(),
            nn.LayerNorm([64, 1, 1024]),  # LayerNorm after third Conv2d layer
            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),  # Input shape: (64, 4, 1024), Output shape: (128, 4, 1024)
            nn.ReLU(),
            nn.Flatten(),  # Flatten the output
            nn.Linear(128 * 1* 1024, 512),

        )
        self.decoder = nn.Sequential(
            nn.Linear(512, 128 * 4 * 8),  # Adjust size to match encoder output
            nn.ReLU(),
            nn.Unflatten(dim=1, unflattened_size=(128, 4, 8)),
            nn.ConvTranspose2d(128, 64, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(32, 16, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(16, 4, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.Conv2d(4, 1, kernel_size=3, stride=1, padding=1),
            nn.Flatten(),
            nn.Linear(64 * 128, 1 * 1 * 1024),
            nn.Unflatten(dim=1, unflattened_size=(1, 1, 1024)),  # Adjust the linear layer to match the reshaped size
            nn.Sigmoid()  # Sigmoid activation for reconstruction
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded

# Instantiate the Convolutional Autoencoder model and move it to GPU
autoencoder = ConvAutoencoder().to(device)

# Check if saved model files exist and load them
encoder_path = folder_path + '/encoder_model.pth'
model_path = folder_path + '/model.pth'

if os.path.exists(encoder_path) and os.path.exists(model_path):
    autoencoder.load_state_dict(torch.load(model_path))
    print("Saved model loaded successfully.")

# Define loss function (reconstruction loss) and optimizer
criterion = nn.MSELoss()
optimizer = AdamW(autoencoder.parameters(), lr=1e-3)  # AdamW optimizer with lr=1e-6
scheduler = CosineAnnealingLR(optimizer, T_max=20)  # CosineAnnealingLR scheduler with T_max=20

# Create a DataLoader for batch training
batch_size = 32
dataset = TensorDataset(tensor_data)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# Training loop
num_epochs = 100  # Increased to 100 epochs
min_loss = float('inf')  # Initialize min_loss with infinity

for epoch in range(num_epochs):
    running_loss = 0.0
    for data in dataloader:
        inputs = data[0].unsqueeze(1).to(device)  # Add a channel dimension and move to GPU
        optimizer.zero_grad()
        outputs = autoencoder(inputs)
        loss = criterion(outputs, inputs)
        loss.backward()
        optimizer.step()
        scheduler.step()
        running_loss += loss.item()

    epoch_loss = running_loss / len(dataloader)
    print(f"Epoch {epoch+1}, Loss: {epoch_loss}")

    # Save the model if the current epoch's loss is lower than the previous min_loss
    if epoch_loss < min_loss:
        min_loss = epoch_loss
        torch.save(autoencoder.state_dict(), model_path)
        torch.save(autoencoder.encoder.state_dict(), encoder_path)

# Save only the encoder's state dictionary
torch.save(autoencoder.encoder.state_dict(), encoder_path)


Saved model loaded successfully.
Epoch 1, Loss: 0.5997831063716154
Epoch 2, Loss: 0.5972721772020626
Epoch 3, Loss: 0.5969053188134601
Epoch 4, Loss: 0.5965129094492067
Epoch 5, Loss: 0.5960837304011944
Epoch 6, Loss: 0.5957320214942398
Epoch 7, Loss: 0.5953963794621611
Epoch 8, Loss: 0.5951170015072065
Epoch 9, Loss: 0.5949037970852961
Epoch 10, Loss: 0.5946683209101772
Epoch 11, Loss: 0.5944855899334264
Epoch 12, Loss: 0.5943301198236514
Epoch 13, Loss: 0.5941750861312722
Epoch 14, Loss: 0.5940288018282941
Epoch 15, Loss: 0.5939165726255705
Epoch 16, Loss: 0.5938024219563067
Epoch 17, Loss: 0.5936924540168816
Epoch 18, Loss: 0.5936029988554373
Epoch 19, Loss: 0.5935064554601269
Epoch 20, Loss: 0.593431177385282
Epoch 21, Loss: 0.5933512090321565
Epoch 22, Loss: 0.5932837714629078
Epoch 23, Loss: 0.5932135930856585
Epoch 24, Loss: 0.5931604472094739
Epoch 25, Loss: 0.5930973406274314
Epoch 26, Loss: 0.5930431123912373
Epoch 27, Loss: 0.5929904505852212
Epoch 28, Loss: 0.59293601116060

KeyboardInterrupt: 

In [ ]:
print(len(dataloader))

1547


In [7]:
def rmse(predictions, targets):
    """
    Calculate the Root Mean Squared Error (RMSE) between two vectors.

    Args:
    - predictions: numpy array, predicted values
    - targets: numpy array, actual values

    Returns:
    - rmse_value: float, RMSE between predictions and targets
    """
    rmse_value = np.sqrt(np.mean((predictions - targets)**2))
    return rmse_value
device='cpu'
for i in range(100):
  L=np.array(tensor_data[i].reshape(1024).to(device))
  K=tensor_data[i].reshape(1,1,1,1024)
  M=autoencoder.forward(K).reshape(1024).cpu().detach().numpy()
  print(L.shape,M.shape)
  print(rmse(L,M))

(1024,) (1024,)
0.76333773
(1024,) (1024,)
0.7632493
(1024,) (1024,)
0.77183175
(1024,) (1024,)
0.7681788
(1024,) (1024,)
0.7723526
(1024,) (1024,)
0.7644661
(1024,) (1024,)
0.76968
(1024,) (1024,)
0.77044624
(1024,) (1024,)
0.7838846
(1024,) (1024,)
0.7726134
(1024,) (1024,)
0.781724
(1024,) (1024,)
0.77280045
(1024,) (1024,)
0.7667094
(1024,) (1024,)
0.77185094
(1024,) (1024,)
0.7660133
(1024,) (1024,)
0.7801126
(1024,) (1024,)
0.7724222
(1024,) (1024,)
0.76468253
(1024,) (1024,)
0.76243806
(1024,) (1024,)
0.76106554
(1024,) (1024,)
0.7858417
(1024,) (1024,)
0.7758244
(1024,) (1024,)
0.77281666
(1024,) (1024,)
0.7685173
(1024,) (1024,)
0.77889085
(1024,) (1024,)
0.7690104
(1024,) (1024,)
0.7655658
(1024,) (1024,)
0.7805065
(1024,) (1024,)
0.7691672
(1024,) (1024,)
0.7694624
(1024,) (1024,)
0.76698846
(1024,) (1024,)
0.7687054
(1024,) (1024,)
0.7682638
(1024,) (1024,)
0.78017384
(1024,) (1024,)
0.7642135
(1024,) (1024,)
0.78883725
(1024,) (1024,)
0.7647839
(1024,) (1024,)
0.7684926
(1